# ML-09 - Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hisham-Walid/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the Week-5 missed-click opportunity model. It contrasts a row-random holdout with a client-grouped holdout, checks every production feature against the prediction timeline, deliberately tests the leakage alarm, and inspects anonymized failure cases. All conclusions are descriptive decision-support for this March 2026 sample; they are not causal claims.

## 1. Two paper findings + my methodology questions

Source: FlyRank, *The State of AI-Driven SEO in Numbers* (March 2026), [public PDF](https://github.com/flyrank-bih/flyrank-ml-internship-starter/blob/main/docs/flyrank-seo-research-march-2026.pdf).

### Finding A - the freshness multiplier (pp. 9 and 36)

The paper reports that mature pages refreshed within 30 days had **3.2x higher health** and **57x more impressions** than mature pages that were not recently refreshed. The paper appropriately presents this as an observed portfolio pattern and discloses that the study is observational.

**Methodology question:** How would the estimated refresh difference change after matching or stratifying refreshed and untouched pages on their pre-refresh impressions, position, client, content age, and reason for selection? The exposure is recent refresh status, while the reported outcomes include a composite health score and impressions from the same rolling snapshot. Editors may preferentially refresh pages that already have demand, and the health score itself includes impressions, position, CTR, and scroll depth. A time-aware comparison using pre-refresh baselines and a matched untreated group would better support a statement about lift; the current design supports a directional association, not a causal refresh effect.

### Finding B - 71% holdout accuracy for growth classification (pp. 5, 29, and 36)

The exploratory appendix reports 71% holdout accuracy for a logistic model that separates growing from declining pages. The label appears to come from 30-day versus previous-30-day impression change: up above +10%, down below -10%, with stable, flat, and new pages defined separately.

**Methodology question:** Was the 80/20 holdout grouped by client or split by row, which label classes entered the binary sample, and what were the held-out class balance, confusion matrix, and confidence interval? Pages from the same brand can share demand and measurement patterns, so a row-random holdout may be optimistic. Accuracy also needs the majority-class base rate beside it. A client-grouped or forward-time test would more directly support generalization to an unseen portfolio or future month. This is a request for validation detail, not a dismissal of the paper's explicitly exploratory result.

In [1]:
import pandas as pd
from IPython.display import display

paper_audit = pd.DataFrame([
    {
        'finding': 'Freshness multiplier',
        'label_or_outcome': 'recent refresh status; health score and impressions',
        'validation_question': 'matched or time-aware pre/post comparison?',
        'safe_evidence_level': 'observed association',
    },
    {
        'finding': '71% growth classification accuracy',
        'label_or_outcome': '30d vs prior-30d impression trend class',
        'validation_question': 'client-grouped or forward-time holdout, with base rate?',
        'safe_evidence_level': 'exploratory discrimination',
    },
])
display(paper_audit)

,finding,label_or_outcome,validation_question,safe_evidence_level
0,Freshness multiplier,recent refresh status; health score and impres...,matched or time-aware pre/post comparison?,observed association
1,71% growth classification accuracy,30d vs prior-30d impression trend class,"client-grouped or forward-time holdout, with b...",exploratory discrimination


## 2. My model under an honest split (before/after)

**Decision timeline:** features use March 1-20 only; labels use March 21-31 only. June remains untouched. The Week-5 model already used a client-grouped holdout; this notebook reconstructs a row-random “before” comparator so the effect of the validation choice is visible.

- **Before:** seeded 75/25 row-random split. A client can appear on both sides, so this estimates performance on more pages from known clients.
- **After:** seeded 75/25 `GroupShuffleSplit` by client. Test clients are absent from training, so this estimates transfer to unseen clients.
- **Primary metric:** median client NDCG@20, because the decision is a ranked review queue within each client.
- **Context metrics:** mean client NDCG@20, mean client precision@20, MAE, and the positive-label base rate. The score gap is descriptive; the grouped result is the claim-bearing result.

In [2]:
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, ndcg_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

RANDOM_SEED = 42
FEATURE_START = '2026-03-01'
DECISION_DATE = '2026-03-20'
OUTCOME_START = '2026-03-21'
OUTCOME_END = '2026-03-31'
MIN_CLIENT_ROWS = 100
K = 20

repo_root = Path.cwd().resolve()
while not (repo_root / 'work' / 'notebooks').exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError('Run this notebook from somewhere inside the repository.')
    repo_root = repo_root.parent
output_dir = repo_root / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
local_file = os.getenv('FLYRANK_MARCH_PARQUET')
hf_token = os.getenv('HF_TOKEN')
if local_file and Path(local_file).is_file():
    escaped = Path(local_file).as_posix().replace("'", "''")
    FACT = f"read_parquet('{escaped}')"
    source_mode = 'authenticated local cache'
elif hf_token:
    escaped_token = hf_token.replace("'", "''")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
    FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
    source_mode = 'authenticated Hugging Face stream'
else:
    raise RuntimeError('Set HF_TOKEN or FLYRANK_MARCH_PARQUET; never paste a token into this notebook.')

feature_query = f"""
WITH daily AS (
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_sum_position
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
), prior AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS prior_impressions,
           SUM(gsc_clicks) AS prior_clicks,
           SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS prior_avg_position,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS prior_active_days,
           100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS prior_ctr_pct
    FROM daily
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
), outcome AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS future_impressions,
           SUM(gsc_clicks) AS future_clicks,
           SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS future_avg_position,
           100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS future_ctr_pct
    FROM daily
    WHERE report_date BETWEEN DATE '{OUTCOME_START}' AND DATE '{OUTCOME_END}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
)
SELECT p.*, o.* EXCLUDE (client_hash_id, content_hash_id)
FROM prior p
JOIN outcome o USING (client_hash_id, content_hash_id)
"""
model_frame = con.execute(feature_query).df()
client_counts = model_frame.groupby('client_hash_id').size()
eligible_clients = client_counts[client_counts >= MIN_CLIENT_ROWS].index
model_frame = model_frame.loc[model_frame['client_hash_id'].isin(eligible_clients)].copy()
model_frame = model_frame.sort_values(['client_hash_id', 'content_hash_id']).reset_index(drop=True)

feature_columns = [
    'prior_impressions', 'prior_clicks', 'prior_avg_position',
    'prior_active_days', 'prior_ctr_pct',
]
forbidden = {
    'client_hash_id', 'content_hash_id', 'future_impressions', 'future_clicks',
    'future_avg_position', 'future_ctr_pct', 'future_position_band',
    'expected_future_ctr_pct', 'future_ctr_gap_pp', 'future_missed_clicks',
    'trend_pct', 'trend_direction', 'is_declining_label',
}
assert model_frame[['client_hash_id', 'content_hash_id']].duplicated().sum() == 0
assert set(feature_columns).isdisjoint(forbidden)
assert model_frame[feature_columns].notna().all().all()
assert pd.Timestamp(DECISION_DATE) < pd.Timestamp(OUTCOME_START)
print(f'Warehouse ready via {source_mode}: {len(model_frame):,} pages across {model_frame.client_hash_id.nunique()} clients.')
print(f'Feature window: {FEATURE_START} to {DECISION_DATE}; outcome: {OUTCOME_START} to {OUTCOME_END}.')

Warehouse ready via authenticated local cache: 86,574 pages across 22 clients.
Feature window: 2026-03-01 to 2026-03-20; outcome: 2026-03-21 to 2026-03-31.


In [3]:
position_bins = [-np.inf, 3, 10, 20, 50, np.inf]
position_labels = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
model_frame['future_position_band'] = pd.cut(
    model_frame['future_avg_position'], bins=position_bins, labels=position_labels
)

random_train_idx, random_test_idx = train_test_split(
    np.arange(len(model_frame)), test_size=0.25, random_state=RANDOM_SEED
)
group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
group_train_idx, group_test_idx = next(
    group_splitter.split(model_frame, groups=model_frame['client_hash_id'])
)

def add_training_defined_target(train_frame, test_frame):
    train_frame = train_frame.copy()
    test_frame = test_frame.copy()
    train_peer = (
        train_frame.groupby('future_position_band', observed=True)
        .agg(impressions=('future_impressions', 'sum'), clicks=('future_clicks', 'sum'))
    )
    train_peer['expected_future_ctr_pct'] = 100 * train_peer['clicks'] / train_peer['impressions']
    for frame in (train_frame, test_frame):
        frame['expected_future_ctr_pct'] = (
            frame['future_position_band'].map(train_peer['expected_future_ctr_pct']).astype(float)
        )
        frame['future_ctr_gap_pp'] = (
            frame['expected_future_ctr_pct'] - frame['future_ctr_pct']
        ).clip(lower=0)
        frame['future_missed_clicks'] = (
            frame['future_impressions'] * frame['future_ctr_gap_pp'] / 100
        )
    return train_frame, test_frame

def fit_and_score(split_name, train_idx, test_idx, extra_features=None):
    train_frame, test_frame = add_training_defined_target(
        model_frame.iloc[train_idx], model_frame.iloc[test_idx]
    )
    used_features = feature_columns + (extra_features or [])
    dummy = DummyRegressor(strategy='mean')
    dummy.fit(train_frame[used_features], train_frame['future_missed_clicks'])
    test_frame['dummy_score'] = dummy.predict(test_frame[used_features])
    fitted = RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=20,
        max_features=0.8, random_state=RANDOM_SEED, n_jobs=-1,
    )
    fitted.fit(train_frame[used_features], np.log1p(train_frame['future_missed_clicks']))
    test_frame['model_score'] = np.expm1(fitted.predict(test_frame[used_features])).clip(min=0)

    def client_metrics(score_column):
        ndcgs, precisions = [], []
        for _, group in test_frame.groupby('client_hash_id'):
            ndcgs.append(ndcg_score(
                [group['future_missed_clicks'].to_numpy()],
                [group[score_column].to_numpy()], k=K,
            ))
            top_k = group.nlargest(min(K, len(group)), score_column)
            precisions.append((top_k['future_missed_clicks'] > 0).mean())
        return np.asarray(ndcgs), np.asarray(precisions)

    model_ndcg, model_precision = client_metrics('model_score')
    dummy_ndcg, _ = client_metrics('dummy_score')
    summary = {
        'split': split_name,
        'train_pages': len(train_frame),
        'test_pages': len(test_frame),
        'train_clients': train_frame['client_hash_id'].nunique(),
        'test_clients': test_frame['client_hash_id'].nunique(),
        'client_overlap': len(set(train_frame['client_hash_id']) & set(test_frame['client_hash_id'])),
        'positive_base_rate': (test_frame['future_missed_clicks'] > 0).mean(),
        'dummy_median_client_ndcg_at_20': np.median(dummy_ndcg),
        'model_median_client_ndcg_at_20': np.median(model_ndcg),
        'model_mean_client_ndcg_at_20': np.mean(model_ndcg),
        'model_mean_client_precision_at_20': np.mean(model_precision),
        'model_mae': mean_absolute_error(test_frame['future_missed_clicks'], test_frame['model_score']),
    }
    return summary, fitted, train_frame, test_frame, model_ndcg

random_result, random_model, random_train, random_test, random_client_ndcg = fit_and_score(
    'Before: row-random', random_train_idx, random_test_idx
)
group_result, group_model, group_train, group_test, group_client_ndcg = fit_and_score(
    'After: client-grouped', group_train_idx, group_test_idx
)
comparison = pd.DataFrame([random_result, group_result])

rng = np.random.default_rng(RANDOM_SEED)
bootstrap_medians = np.array([
    np.median(rng.choice(group_client_ndcg, size=len(group_client_ndcg), replace=True))
    for _ in range(2000)
])
group_ci_low, group_ci_high = np.quantile(bootstrap_medians, [0.025, 0.975])

display(comparison.style.format({
    'positive_base_rate': '{:.1%}',
    'dummy_median_client_ndcg_at_20': '{:.3f}',
    'model_median_client_ndcg_at_20': '{:.3f}',
    'model_mean_client_ndcg_at_20': '{:.3f}',
    'model_mean_client_precision_at_20': '{:.1%}',
    'model_mae': '{:.3f}',
}))
print(f'Grouped median client NDCG@20 bootstrap 95% interval: [{group_ci_low:.3f}, {group_ci_high:.3f}]')
print('The grouped score is the claim-bearing estimate; the row-random score is shown only to audit validation-design sensitivity.')

,split,train_pages,test_pages,train_clients,test_clients,client_overlap,positive_base_rate,dummy_median_client_ndcg_at_20,model_median_client_ndcg_at_20,model_mean_client_ndcg_at_20,model_mean_client_precision_at_20,model_mae
0,Before: row-random,64930,21644,22,22,22,72.4%,0.082,0.733,0.708,87.0%,0.865
1,After: client-grouped,76321,10253,16,6,0,62.6%,0.044,0.797,0.693,85.0%,0.430


Grouped median client NDCG@20 bootstrap 95% interval: [0.405, 0.876]
The grouped score is the claim-bearing estimate; the row-random score is shown only to audit validation-design sensitivity.


## 3. Leakage audit and real failure examples

The production features are IDs-free, product-score-free summaries known by the March 20 decision time. The target is constructed only from March 21-31 outcomes. Client and content hashes are retained solely for grouping and joins. The population does require a page to have at least one outcome-window impression because the target needs an observed future CTR; this is a disclosed survivorship condition, so the result does **not** cover pages that disappear completely in the outcome window.

The row-random comparator should show client overlap; the grouped split must show zero. A deliberate test-only leak (`future_impressions` plus `future_ctr_gap_pp`, both outcome-derived target components) checks that the audit harness reacts. Those probe columns are never production features. A large score jump confirms why the temporal and column allowlists matter.

In [4]:
feature_audit = pd.DataFrame([
    ('prior_impressions', 'March 1-20 GSC aggregate', True, False, False, False),
    ('prior_clicks', 'March 1-20 GSC aggregate', True, False, False, False),
    ('prior_avg_position', 'March 1-20 GSC aggregate', True, False, False, False),
    ('prior_active_days', 'March 1-20 GSC aggregate', True, False, False, False),
    ('prior_ctr_pct', 'March 1-20 clicks / impressions', True, False, False, False),
], columns=[
    'feature', 'provenance', 'known_at_decision', 'overlaps_outcome',
    'label_derived', 'product_flag_or_score',
])
assert feature_audit['known_at_decision'].all()
assert not feature_audit[['overlaps_outcome', 'label_derived', 'product_flag_or_score']].any().any()
assert group_result['client_overlap'] == 0
assert random_result['client_overlap'] > 0

leak_result, _, _, _, _ = fit_and_score(
    'Test-only deliberate leak',
    group_train_idx,
    group_test_idx,
    extra_features=['future_impressions', 'future_ctr_gap_pp'],
)
leak_probe = pd.DataFrame([
    {
        'feature_set': 'production features',
        'median_client_ndcg_at_20': group_result['model_median_client_ndcg_at_20'],
        'allowed_for_claim': True,
    },
    {
        'feature_set': 'production + two outcome-derived components',
        'median_client_ndcg_at_20': leak_result['model_median_client_ndcg_at_20'],
        'allowed_for_claim': False,
    },
])
leak_jump = (
    leak_result['model_median_client_ndcg_at_20']
    - group_result['model_median_client_ndcg_at_20']
)
assert leak_jump > 0.05, 'Leakage probe did not move enough; inspect the validation harness.'

importance = pd.DataFrame({
    'feature': feature_columns,
    'model_importance': group_model.feature_importances_,
}).sort_values('model_importance', ascending=False, ignore_index=True)

group_test = group_test.copy()
group_test['absolute_error'] = (
    group_test['future_missed_clicks'] - group_test['model_score']
).abs()
group_test['error_direction'] = np.where(
    group_test['model_score'] < group_test['future_missed_clicks'],
    'under-prediction', 'over-prediction',
)
worst_cases = group_test.nlargest(5, 'absolute_error').copy()
worst_cases.insert(0, 'case', [f'case_{i}' for i in range(1, 6)])
error_columns = [
    'case', 'prior_impressions', 'prior_clicks', 'prior_avg_position',
    'prior_active_days', 'prior_ctr_pct', 'future_impressions',
    'future_ctr_pct', 'future_missed_clicks', 'model_score',
    'absolute_error', 'error_direction',
]

display(feature_audit)
print('Leakage alarm test (second row is intentionally invalid):')
display(leak_probe.style.format({'median_client_ndcg_at_20': '{:.3f}'}))
print(f'Deliberate-leak NDCG@20 jump: {leak_jump:.3f}')
print('Production feature importance (model behavior, not causation):')
display(importance.style.format({'model_importance': '{:.3f}'}))
print('Five largest grouped-holdout errors (identifiers intentionally omitted):')
display(worst_cases[error_columns].style.format({
    'prior_avg_position': '{:.2f}', 'prior_ctr_pct': '{:.3f}',
    'future_ctr_pct': '{:.3f}', 'future_missed_clicks': '{:.2f}',
    'model_score': '{:.2f}', 'absolute_error': '{:.2f}',
}))

,feature,provenance,known_at_decision,overlaps_outcome,label_derived,product_flag_or_score
0,prior_impressions,March 1-20 GSC aggregate,True,False,False,False
1,prior_clicks,March 1-20 GSC aggregate,True,False,False,False
2,prior_avg_position,March 1-20 GSC aggregate,True,False,False,False
3,prior_active_days,March 1-20 GSC aggregate,True,False,False,False
4,prior_ctr_pct,March 1-20 clicks / impressions,True,False,False,False


Leakage alarm test (second row is intentionally invalid):


,feature_set,median_client_ndcg_at_20,allowed_for_claim
0,production features,0.797,True
1,production + two outcome-derived components,1.000,False


Deliberate-leak NDCG@20 jump: 0.203
Production feature importance (model behavior, not causation):


,feature,model_importance
0,prior_impressions,0.535
1,prior_ctr_pct,0.315
2,prior_avg_position,0.104
3,prior_clicks,0.031
4,prior_active_days,0.015


Five largest grouped-holdout errors (identifiers intentionally omitted):


,case,prior_impressions,prior_clicks,prior_avg_position,prior_active_days,prior_ctr_pct,future_impressions,future_ctr_pct,future_missed_clicks,model_score,absolute_error,error_direction
63970,case_1,51151.000000,2.000000,7.97,20,0.004,38181.000000,0.005,108.48,34.69,73.79,under-prediction
5251,case_2,639.000000,3.000000,3.19,18,0.469,19381.000000,0.021,52.08,0.42,51.67,under-prediction
5311,case_3,3171.000000,0.000000,3.84,4,0.000,14446.000000,0.014,39.80,5.12,34.68,under-prediction
66567,case_4,10433.000000,3.000000,6.13,20,0.029,10308.000000,0.010,28.83,10.58,18.25,under-prediction
4367,case_5,14757.000000,48.000000,5.77,20,0.325,15082.000000,0.166,18.64,2.36,16.28,under-prediction


### Failure interpretation

The largest absolute misses are inspected as cases, not anecdotes of model success. They reveal where future missed-click opportunity can be much larger than the compressed predictions of a depth-limited forest. High-volume pages can change sharply over the short outcome window, while the five historical summaries contain no query mix, page change, seasonality, or intervention signal. The model is therefore suited to **decision-support prioritization**, not a forecast of exact recovered clicks. Before operational use, I would add more historical windows, test stability over several forward months, and monitor errors by client and traffic scale.

## 4. Claim rewrite

**Week-5 wording to tighten:** “The model wins the primary NDCG metric.”

**Public-safe rewrite:** “In this March 2026 sample, the fixed Random Forest measured the grouped-holdout median client NDCG@20 shown above on six unseen clients. Its ranking result is directional decision-support, not evidence that editing the selected pages will cause click gains. The small number of held-out clients, the bootstrap interval, outcome-window survivorship, and the error cases limit generalization; a multi-month forward test is required before a broader performance claim.”

The row-random number is not used as evidence of unseen-client performance. Feature importance describes what the fitted model used; it does not identify causal optimization levers. The deliberate-leak score is only a validation alarm check and is excluded from every model claim.

In [5]:
metrics = {
    'assignment': 'ML-09',
    'seed': RANDOM_SEED,
    'feature_window': [FEATURE_START, DECISION_DATE],
    'outcome_window': [OUTCOME_START, OUTCOME_END],
    'population_condition': 'prior impressions >=100; client rows >=100; at least one outcome-window impression',
    'model_rows': int(len(model_frame)),
    'comparison': comparison.round(6).to_dict(orient='records'),
    'grouped_median_client_ndcg_at_20_bootstrap_95pct': [
        round(float(group_ci_low), 6), round(float(group_ci_high), 6)
    ],
    'deliberate_leak_probe': {
        'invalid_features': ['future_impressions', 'future_ctr_gap_pp'],
        'median_client_ndcg_at_20': round(float(leak_result['model_median_client_ndcg_at_20']), 6),
        'absolute_jump_vs_production': round(float(leak_jump), 6),
        'allowed_for_claim': False,
    },
    'production_feature_importance': importance.round(6).to_dict(orient='records'),
    'versions': {
        'scikit_learn': sklearn.__version__,
        'pandas': pd.__version__,
        'duckdb': duckdb.__version__,
    },
}
metrics_path = output_dir / 'ml09_validation_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
print('Wrote reproducible metrics receipt to work/outputs/ml09_validation_metrics.json.')

Wrote reproducible metrics receipt to work/outputs/ml09_validation_metrics.json.


## 5. Self-check

- [x] Two paper findings are named, sourced, and questioned constructively.
- [x] The same fixed model is shown before and after the validation-design improvement.
- [x] The grouped holdout has zero client overlap; the row-random comparator exposes overlap.
- [x] Feature and timeline audits exclude IDs, outcome siblings, labels, and product scores.
- [x] A deliberate invalid-feature probe proves the leakage alarm responds, then is excluded.
- [x] Base rate, client-level ranking metrics, uncertainty, and identifier-free failure cases are reported.
- [x] Claims use observed, measured, directional, and decision-support language.
- [x] The notebook runs top to bottom, writes a metrics receipt, and prints no client names, URLs, or private queries.

**Remaining limitation:** this notebook audits one mid-panel month and six unseen test clients. It does not touch the sealed June partition and does not estimate the causal effect of acting on a recommendation.